# ALand-AI Fine-tune Qwen2.5-0.5B (494M params)
**Sebelum mulai:** Runtime → Change runtime type → **T4 GPU**

Model hasil training diupload otomatis ke GitHub Releases.

In [ ]:
# ── KONFIGURASI — isi bagian ini ──
GITHUB_TOKEN = 'ISI_TOKEN_KAMU_DISINI'  # ghp_...
GITHUB_REPO  = 'FerdiKusmanto/AI-EVIL-BY-ALAND'
# ─────────────────────────────────
print('Config OK')

In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes huggingface_hub
!apt-get install -qq git-lfs
print('Deps installed ✅')

In [ ]:
# Download dataset 300MB dari GitHub Releases
import subprocess, json, os
os.makedirs('/content/data', exist_ok=True)
!curl -L -o /content/data/dataset.jsonl.gz \
  https://github.com/FerdiKusmanto/AI-EVIL-BY-ALAND/releases/download/dataset-latest/dataset.jsonl.gz
!gunzip -f /content/data/dataset.jsonl.gz
# Hitung jumlah data
count = sum(1 for _ in open('/content/data/dataset.jsonl'))
size  = os.path.getsize('/content/data/dataset.jsonl') / 1024 / 1024
print(f'Dataset: {count} pairs, {size:.1f}MB')

In [ ]:
from datasets import Dataset
import json, random

data = []
with open('/content/data/dataset.jsonl', encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            if len(obj.get('messages', [])) >= 2:
                data.append(obj)
        except: pass

random.shuffle(data)
data = data[:80000]  # 80k pairs untuk T4 16GB
dataset = Dataset.from_list(data)
print(f'Training dengan {len(data)} pairs')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'  # 494M parameter

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True
)
print(f'Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params ✅')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64,
    target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
))
model.print_trainable_parameters()

In [ ]:
def fmt(ex):
    text = tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    tok  = tokenizer(text, truncation=True, max_length=512, padding='max_length')
    tok['labels'] = tok['input_ids'].copy()
    return tok

tokenized = dataset.map(fmt, remove_columns=['messages'], num_proc=2)
print('Tokenized ✅')

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir='/content/aland-lora',
        num_train_epochs=3,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=2e-4, fp16=True,
        logging_steps=50, save_steps=500,
        warmup_ratio=0.05, report_to='none',
        dataloader_num_workers=2,
    ),
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()
print('Training selesai ✅')

In [ ]:
# Merge LoRA + simpan
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, '/content/aland-lora').merge_and_unload()
merged.save_pretrained('/content/aland-merged')
tokenizer.save_pretrained('/content/aland-merged')
print('Merge selesai ✅')

In [ ]:
# Convert ke GGUF Q4_K_M
!git clone -q https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q gguf
!python /content/llama.cpp/convert_hf_to_gguf.py /content/aland-merged \
    --outfile /content/aland-id-500M-q4.gguf --outtype q4_k_m

import os
size = os.path.getsize('/content/aland-id-500M-q4.gguf') / 1024 / 1024
print(f'GGUF: {size:.0f}MB ✅')

In [ ]:
# Upload ke GitHub Releases
import subprocess
env = {**__import__('os').environ, 'GH_TOKEN': GITHUB_TOKEN}

# Install gh CLI
subprocess.run('curl -sS https://webi.sh/gh | sh', shell=True, capture_output=True)
subprocess.run('export PATH="$HOME/.local/bin:$PATH"', shell=True)

result = subprocess.run(
    f'~/.local/bin/gh release upload model-500M /content/aland-id-500M-q4.gguf --clobber '
    f'--repo {GITHUB_REPO} 2>/dev/null || '
    f'~/.local/bin/gh release create model-500M /content/aland-id-500M-q4.gguf '
    f'--title "Model 500M" --notes "Qwen2.5-0.5B fine-tuned" --repo {GITHUB_REPO}',
    shell=True, env=env, capture_output=True, text=True
)
print(result.stdout or result.stderr)
print(f'✅ Download: https://github.com/{GITHUB_REPO}/releases/download/model-500M/aland-id-500M-q4.gguf')
print(f'\nDi laptop: aland-ai pull {GITHUB_REPO}/model-500M/aland-id-500M-q4.gguf')